# Pool-Based Sampling in Active Learning - Starter Notebook
This notebook demonstrates classical pool-based active learning where a model repeatedly selects the most informative instances from an unlabeled pool.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

In [ ]:
X, y = make_classification(n_samples=1400, n_features=10, n_informative=7, n_redundant=2, random_state=42)
X_pool, X_test, y_pool, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

rng = np.random.default_rng(42)
labeled_mask = np.zeros(len(X_pool), dtype=bool)
labeled_mask[rng.choice(len(X_pool), size=35, replace=False)] = True

In [ ]:
history = []
for step in range(9):
    model = LogisticRegression(max_iter=1000)
    model.fit(X_pool[labeled_mask], y_pool[labeled_mask])
    pred_test = model.predict(X_test)
    history.append({'step': step, 'labeled': labeled_mask.sum(), 'test_accuracy': accuracy_score(y_test, pred_test)})

    unlabeled = np.where(~labeled_mask)[0]
    probs = model.predict_proba(X_pool[unlabeled])
    entropy = -np.sum(probs * np.log(probs + 1e-9), axis=1)
    query_ids = unlabeled[np.argsort(entropy)[-25:]]
    labeled_mask[query_ids] = True

pd.DataFrame(history).round(3)